# ResUpNet Phase 2 Training Report

This notebook is for reporting real training and evaluation outputs. It reads `history.json`, checkpoints, and evaluation files produced by the PyTorch ResUpNet pipeline.

Do not edit metric values manually for paper/report use. If a run has not finished, leave the report incomplete and mark it as pending.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

# Change this to the run you want to report.
RUN_DIR = Path(r"E:\\ResUpNet\\runs\\resupnet_pilot_balanced_600")
EVAL_DIR = RUN_DIR / "evaluation_tta_post"

history_path = RUN_DIR / "logs" / "history.json"
config_path = RUN_DIR / "run_config.json"
summary_path = EVAL_DIR / "evaluation_summary.json"

print("Run dir:", RUN_DIR)
print("History exists:", history_path.exists())
print("Config exists:", config_path.exists())
print("Evaluation summary exists:", summary_path.exists())

In [ ]:
if not history_path.exists():
    raise FileNotFoundError(f"Missing real training history: {history_path}")

history = pd.read_json(history_path)
display(history.tail())

metric_cols = [
    "train_loss", "train_dice", "train_iou", "train_precision", "train_recall", "train_f1", "train_specificity", "train_accuracy",
    "val_loss", "val_dice", "val_iou", "val_precision", "val_recall", "val_f1", "val_specificity", "val_accuracy",
]
available = [c for c in metric_cols if c in history.columns]
best_row = history.loc[history["val_dice"].idxmax()]
print("Best epoch by validation Dice:")
display(best_row[["epoch", *available]].to_frame("value"))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
plots = [
    ("loss", "train_loss", "val_loss"),
    ("dice", "train_dice", "val_dice"),
    ("iou", "train_iou", "val_iou"),
    ("precision", "train_precision", "val_precision"),
    ("recall", "train_recall", "val_recall"),
    ("f1", "train_f1", "val_f1"),
    ("specificity", "train_specificity", "val_specificity"),
    ("accuracy", "train_accuracy", "val_accuracy"),
]
for ax, (title, train_col, val_col) in zip(axes.ravel(), plots):
    if train_col in history.columns:
        ax.plot(history["epoch"], history[train_col], label="train")
    if val_col in history.columns:
        ax.plot(history["epoch"], history[val_col], label="validation")
    ax.set_title(title.upper())
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()

In [ ]:
config = json.loads(config_path.read_text()) if config_path.exists() else {}
report_config = {
    "backend": config.get("backend"),
    "device": config.get("device"),
    "gpu": config.get("cuda_device_name"),
    "input_shape": config.get("input_shape"),
    "train_shape": config.get("train_shape"),
    "val_shape": config.get("val_shape"),
    "batch_size": config.get("batch_size"),
    "base_filters": config.get("base_filters"),
    "learning_rate": config.get("learning_rate"),
    "mixed_precision": config.get("mixed_precision"),
    "balanced_batches": config.get("balanced_batches"),
}
display(pd.Series(report_config, name="training_config"))

In [ ]:
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    rows = []
    for group in ["all_test_rows", "tumor_test_rows", "empty_true_test_rows"]:
        group_data = summary.get(group, {})
        row = {"group": group, "count": group_data.get("count")}
        for metric in ["dice", "iou", "precision", "recall", "f1", "specificity", "hd95", "asd"]:
            if metric in group_data:
                row[f"{metric}_mean"] = group_data[metric].get("mean")
                row[f"{metric}_std"] = group_data[metric].get("std")
        rows.append(row)
    display(pd.DataFrame(rows))
else:
    print("Evaluation summary is not available yet. Run evaluate_phase2_model_torch.py after training.")

## Reporting Rule

Use the best validation epoch for model selection, then report locked test-set evaluation from `evaluation_summary.json`. Do not tune thresholds or select checkpoints using the test set.